# 01 — Data Exploration
Explore the brain tumor MRI dataset: class distribution, sample images, and preprocessing effects.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import cv2
from collections import Counter

sys.path.append('../src')
from utils.visualization import plot_class_distribution
from utils.preprocessing import preprocess_for_efficientnet, anisotropic_diffusion_filter, normalize, load_mri

DATA_DIR = Path('../data/raw')
CLASSES = ['glioma', 'meningioma', 'no_tumor', 'pituitary']
print('Data directory:', DATA_DIR.resolve())

## 1. Class Distribution

In [ ]:
class_counts = {}
for cls in CLASSES:
    cls_dir = DATA_DIR / 'train' / cls
    if cls_dir.exists():
        class_counts[cls] = len(list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png')))
    else:
        class_counts[cls] = 0

print('Class counts:', class_counts)
plot_class_distribution(class_counts, title='Training Set Class Distribution')

## 2. Sample Images per Class

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, cls in zip(axes, CLASSES):
    cls_dir = DATA_DIR / 'train' / cls
    imgs = list(cls_dir.glob('*.jpg'))[:1] if cls_dir.exists() else []
    if imgs:
        img = cv2.cvtColor(cv2.imread(str(imgs[0])), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
    ax.set_title(cls.replace('_', ' ').title(), fontweight='bold')
    ax.axis('off')
plt.suptitle('Sample MRI Images per Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Preprocessing Effects

In [ ]:
# Show effect of anisotropic filtering on a sample image
sample_path = next((DATA_DIR / 'train' / 'glioma').glob('*.jpg'), None)
if sample_path:
    original = load_mri(str(sample_path))
    norm = normalize(original)
    filtered = anisotropic_diffusion_filter(norm, num_iter=10)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].imshow(original); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(norm); axes[1].set_title('Normalized [0,1]'); axes[1].axis('off')
    axes[2].imshow(filtered); axes[2].set_title('Anisotropic Filtered'); axes[2].axis('off')
    plt.suptitle('Preprocessing Pipeline', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('No sample image found. Check DATA_DIR path.')

## 4. Image Statistics

In [ ]:
heights, widths = [], []
for cls in CLASSES:
    cls_dir = DATA_DIR / 'train' / cls
    if not cls_dir.exists(): continue
    for img_path in list(cls_dir.glob('*.jpg'))[:50]:
        img = cv2.imread(str(img_path))
        if img is not None:
            h, w = img.shape[:2]
            heights.append(h)
            widths.append(w)

print(f'Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}')
print(f'Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}')